In [3]:
from langgraph.graph import StateGraph, START, END
from typing import Annotated, Literal, TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import SecretStr, BaseModel, Field
from langgraph.checkpoint.memory import InMemorySaver
import os 

In [4]:
# === Utility Functions ===
def get_secret(key_name: str) -> SecretStr:
    value = os.getenv(key_name)
    if not value:
        raise ValueError(
            f"❌ {key_name} not found. Please set it in your .env file.")
    return SecretStr(value)

In [5]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.5,
    api_key=get_secret("GROQ_API_KEY"),
)

In [6]:
class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

In [7]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [8]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [9]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({"topic": "Gaming"}, config=config1)

{'topic': 'Gaming',
 'joke': 'Why did the gamer bring a ladder to the tournament?  \n\nBecause they heard the competition was on a whole new level! 🎮😆',
 'explanation': '**Joke:**  \n*Why did the gamer bring a ladder to the tournament?*  \n\n*Because they heard the competition was on a **whole new level**! 🎮😆*\n\n---\n\n## Why it’s funny – a step‑by‑step breakdown\n\n| Element | What it means literally | What it means figuratively (in gaming) | Why the combination is a pun |\n|---------|------------------------|------------------------------------------|------------------------------|\n| **Ladder** | A piece of equipment with rungs that lets you climb up or down. | In many video games a “ladder” can be a literal object you climb, but the word also evokes the idea of “levels” (e.g., moving upward). | The gamer brings a *physical* ladder expecting to climb something. |\n| **Competition was on a whole new level** | “Level” can refer to a floor of a building or a stage in a hierarchy. | In

In [17]:
config2 = {"configurable": {"thread_id": "1"}}
workflow.invoke({"topic": "Bus"}, config=config1)

{'topic': 'Bus',
 'joke': 'Why did the bus apply for a job?\n\nBecause it heard there were *lots of stops* and it wanted to *pick up* some extra *fare* experience! 🚍😄',
 'explanation': '**Explanation of the joke**\n\nThe joke plays on a few everyday words that have two different meanings—one literal (the way a bus works) and one figurative (the language we use about jobs). Let’s break it down piece by piece.\n\n| Phrase in the punchline | Literal meaning (bus‑related) | Figurative meaning (job‑related) |\n|--------------------------|------------------------------|-----------------------------------|\n| **“lots of stops”**      | A bus travels from one stop to the next, picking up and dropping off passengers at many locations. | In a job, “stops” can be thought of as “opportunities” or “places where you can pause and do something useful.” The bus “hears” that the job has many such opportunities. |\n| **“pick up”**            | Buses “pick up” passengers at each stop. | “Pick up” is a co

In [18]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Bus', 'joke': 'Why did the bus apply for a job?\n\nBecause it heard there were *lots of stops* and it wanted to *pick up* some extra *fare* experience! 🚍😄', 'explanation': '**Explanation of the joke**\n\nThe joke plays on a few everyday words that have two different meanings—one literal (the way a bus works) and one figurative (the language we use about jobs). Let’s break it down piece by piece.\n\n| Phrase in the punchline | Literal meaning (bus‑related) | Figurative meaning (job‑related) |\n|--------------------------|------------------------------|-----------------------------------|\n| **“lots of stops”**      | A bus travels from one stop to the next, picking up and dropping off passengers at many locations. | In a job, “stops” can be thought of as “opportunities” or “places where you can pause and do something useful.” The bus “hears” that the job has many such opportunities. |\n| **“pick up”**            | Buses “pick up” passengers at each stop. 

In [19]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Bus', 'joke': 'Why did the bus apply for a job?\n\nBecause it heard there were *lots of stops* and it wanted to *pick up* some extra *fare* experience! 🚍😄', 'explanation': '**Explanation of the joke**\n\nThe joke plays on a few everyday words that have two different meanings—one literal (the way a bus works) and one figurative (the language we use about jobs). Let’s break it down piece by piece.\n\n| Phrase in the punchline | Literal meaning (bus‑related) | Figurative meaning (job‑related) |\n|--------------------------|------------------------------|-----------------------------------|\n| **“lots of stops”**      | A bus travels from one stop to the next, picking up and dropping off passengers at many locations. | In a job, “stops” can be thought of as “opportunities” or “places where you can pause and do something useful.” The bus “hears” that the job has many such opportunities. |\n| **“pick up”**            | Buses “pick up” passengers at each stop.

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Bus', 'joke': 'Why did the bus apply for a job?\n\nBecause it heard there were *lots of stops* and it wanted to *pick up* some extra *fare* experience! 🚍😄', 'explanation': '**Explanation of the joke**\n\nThe joke plays on a few everyday words that have two different meanings—one literal (the way a bus works) and one figurative (the language we use about jobs). Let’s break it down piece by piece.\n\n| Phrase in the punchline | Literal meaning (bus‑related) | Figurative meaning (job‑related) |\n|--------------------------|------------------------------|-----------------------------------|\n| **“lots of stops”**      | A bus travels from one stop to the next, picking up and dropping off passengers at many locations. | In a job, “stops” can be thought of as “opportunities” or “places where you can pause and do something useful.” The bus “hears” that the job has many such opportunities. |\n| **“pick up”**            | Buses “pick up” passengers at each stop.

In [22]:
# Time travel checkpoints
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f0e8943-023c-63e4-800d-087f64670ff7"}})

StateSnapshot(values={'topic': 'Bus', 'joke': 'Why did the bus apply for a job?\n\nBecause it heard there were *lots of stops* and it wanted to *pick up* some extra *fare* experience! 🚍😄', 'explanation': '**What makes the joke funny?**  \n\n1. **Wordplay on “high‑way.”**  \n   - In everyday language a *highway* is just a big, fast road.  \n   - The joke pretends the word is being taken literally: *high* → “up in the air,” *way* → “a path.”  \n   - If the road is “high,” the bus driver would need something to reach that height—hence the ladder.\n\n2. **Pun on “elevated line.”**  \n   - An *elevated line* (or “el”) is a type of train or subway that runs on tracks raised above street level.  \n   - The driver mishears “elevated” as “elevated” in the sense of “lifted up,” again suggesting a need for a ladder.\n\n3. **The visual gag.**  \n   - Imagining a bus driver walking onto a regular city street with a ladder strapped to the bus is absurd and vivid, which adds to the humor.  \n   - The